<a href="https://colab.research.google.com/github/HashamHassan-01/flyrank-ml-internship-hasham/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HashamHassan-01/flyrank-ml-internship-hasham/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector
### Feature Vector

The feature vector contains four observable content and search-performance signals:

- **search_volume**: estimated search demand associated with the page.
- **content_age_days**: age of the content in days.
- **avg_position**: observed average search position.
- **ctr**: observed click-through rate.

These features are used as signals for prioritizing pages for content review. Missing numeric values are handled using median imputation so that the final feature matrix contains no missing values.

In [ ]:
import pandas as pd
import numpy as np

# Load the anonymized FlyRank starter dataset
df = pd.read_csv(
    "https://raw.githubusercontent.com/HashamHassan-01/flyrank-ml-internship-hasham/main/data/raw/content_refresh_anonymized.csv"
)

print("Dataset shape:", df.shape)
display(df.head())

print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7



Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [ ]:
# Define the observable features used for the content opportunity analysis
feature_cols = [
    "search_volume",
    "content_age_days",
    "avg_position",
    "ctr"
]

# Build a copy of the feature matrix
X = df[feature_cols].copy()

# Check missing values before handling them
print("Missing values before handling:")
print(X.isna().sum())

# Fill missing numeric values using the median of each feature
X = X.fillna(X.median())

# Final checks
print("\nFeature vector shape:", X.shape)

print("\nMissing values after handling:")
print(X.isna().sum())

print("\nFeature preview:")
display(X.head())

Missing values before handling:
search_volume       2468
content_age_days       0
avg_position           0
ctr                    0
dtype: int64

Feature vector shape: (30000, 4)

Missing values after handling:
search_volume       0
content_age_days    0
avg_position        0
ctr                 0
dtype: int64

Feature preview:


,search_volume,content_age_days,avg_position,ctr
0,10.0,187,10.6,0.76
1,90.0,445,20.3,0.05
2,0.0,141,36.5,0.09
3,10.0,463,6.2,0.49
4,0.0,263,44.0,0.13


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature notes

| Feature | Meaning | Missing-value handling | Categorical? | Available when? |
|---|---|---|---|---|
| `search_volume` | Estimated search demand associated with the page | 2,468 missing values were filled using the median | No | Available as an observable search signal at the time of prioritization |
| `content_age_days` | Age of the content in days | No missing values | No | Available at the time of prioritization |
| `avg_position` | Observed average search position | No missing values | No | Available as an observed search-performance signal at the time of prioritization |
| `ctr` | Observed click-through rate | No missing values | No | Available as an observed search-performance signal at the time of prioritization |

All four selected features are numeric, so no categorical encoding was required.

The features are used for decision-support prioritization based on observed signals. They are not treated as proof that changing a page will cause an improvement in search performance.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Summarize the selected features

feature_notes = pd.DataFrame({
    "feature": feature_cols,
    "dtype": [str(X[col].dtype) for col in feature_cols],
    "missing_after_handling": [int(X[col].isna().sum()) for col in feature_cols],
    "unique_values": [int(X[col].nunique()) for col in feature_cols]
})

display(feature_notes)

print("\nAll selected features are numeric:")
print(all(pd.api.types.is_numeric_dtype(X[col]) for col in feature_cols))

print("\nFinal missing values:")
print(X.isna().sum())

,feature,dtype,missing_after_handling,unique_values
0,search_volume,float64,0,41
1,content_age_days,int64,0,225
2,avg_position,float64,0,869
3,ctr,float64,0,401



All selected features are numeric:
True

Final missing values:
search_volume       0
content_age_days    0
avg_position        0
ctr                 0
dtype: int64


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### Leakage check

The leakage check focused on three possible problems:

1. **Label-derived features** — fields that directly contain or reconstruct the target.
2. **Future-window features** — fields calculated using information that would only be available after the prediction or prioritization point.
3. **Outcome or product flags** — fields that may encode downstream performance or decisions.

For the later capstone workflow, the Opportunity Score is constructed directly from `search_volume`, `content_age_days`, `avg_position`, and `ctr`. Therefore, these four features can be used to construct and explain the score, but training a model to predict that same constructed score from these same inputs does not demonstrate prediction of an independent real-world outcome.

This was treated as a framing limitation rather than evidence of causal or future-performance prediction.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Columns examined during the leakage and privacy review

leakage_candidates = [
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "trend_direction",
    "trend_pct",
    "position_tier",
    "impression_tier"
]

print("Selected model features:")
print(feature_cols)

print("\nPotential leakage or downstream-performance fields:")
for col in leakage_candidates:
    print("-", col)

# Check that the selected feature vector does not contain
# these potential leakage candidates
overlap = set(feature_cols).intersection(leakage_candidates)

print("\nOverlap between selected features and leakage candidates:")
print(overlap if overlap else "None")

# Explicitly document the constructed-score issue
constructed_score_features = set([
    "search_volume",
    "content_age_days",
    "avg_position",
    "ctr"
])

selected_feature_set = set(feature_cols)

print("\nDo selected features match the inputs used to construct the Opportunity Score?")
print(selected_feature_set == constructed_score_features)

if selected_feature_set == constructed_score_features:
    print(
        "\nLeakage/framing finding: The Opportunity Score is constructed "
        "from the same four features used as model inputs. A model trained "
        "on this target can reproduce the scoring rule, but this does not "
        "represent independent outcome prediction."
    )


Selected model features:
['search_volume', 'content_age_days', 'avg_position', 'ctr']

Potential leakage or downstream-performance fields:
- impressions_last_30d
- clicks_last_30d
- sessions_last_30d
- impressions_prev_30d
- clicks_prev_30d
- sessions_prev_30d
- trend_direction
- trend_pct
- position_tier
- impression_tier

Overlap between selected features and leakage candidates:
None

Do selected features match the inputs used to construct the Opportunity Score?
True

Leakage/framing finding: The Opportunity Score is constructed from the same four features used as model inputs. A model trained on this target can reproduce the scoring rule, but this does not represent independent outcome prediction.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### Excluded fields

The following fields were excluded from the feature vector to reduce leakage risk, avoid downstream or future-window information, and keep the workflow public-safe.

| Excluded field(s) | Why excluded |
|---|---|
| `content_id` | Identifier only; it does not represent a meaningful generalizable signal. |
| `client_id` | Used for grouped validation, not as a predictive feature, to avoid learning client-specific patterns. |
| `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d` | Recent performance-window fields that could create timing or downstream-information concerns. |
| `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d` | Previous performance-window fields were excluded to keep the feature vector focused and avoid mixing additional temporal windows into the scoring workflow. |
| `trend_direction`, `trend_pct` | Derived movement signals that may overlap with downstream performance interpretation. |
| `position_tier`, `impression_tier` | Derived categorical versions of existing performance signals; excluded to avoid redundant representations. |
| `provider_used`, `model_used` | Tool/provider metadata; not required for the content-prioritization question. |
| `engagement_rate`, `scroll_rate`, `ai_traffic_pct` | Additional engagement and traffic outcome signals were excluded to keep the feature vector limited to the four predefined observable signals used in the Opportunity Score. |

The final feature vector therefore contains only `search_volume`, `content_age_days`, `avg_position`, and `ctr`.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Document excluded fields and verify they are not in the feature vector

excluded_fields = [
    "content_id",
    "client_id",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "trend_direction",
    "trend_pct",
    "position_tier",
    "impression_tier",
    "provider_used",
    "model_used",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

print("Excluded fields checked:")
for col in excluded_fields:
    print("-", col)

overlap = set(feature_cols).intersection(excluded_fields)

print("\nOverlap between selected and excluded fields:")
print(overlap if overlap else "None")

print("\nFinal feature vector:")
print(feature_cols)

print("\nPrivacy check for selected features:")
identifier_like = [
    col for col in feature_cols
    if "id" in col.lower() or "url" in col.lower()
]

print(identifier_like if identifier_like else "No identifier or URL fields selected.")

Excluded fields checked:
- content_id
- client_id
- impressions_last_30d
- clicks_last_30d
- sessions_last_30d
- impressions_prev_30d
- clicks_prev_30d
- sessions_prev_30d
- trend_direction
- trend_pct
- position_tier
- impression_tier
- provider_used
- model_used
- engagement_rate
- scroll_rate
- ai_traffic_pct

Overlap between selected and excluded fields:
None

Final feature vector:
['search_volume', 'content_age_days', 'avg_position', 'ctr']

Privacy check for selected features:
No identifier or URL fields selected.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.